In [156]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [157]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [158]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [159]:
from sklearn.model_selection import train_test_split

#split the dataframe into 80%-training and 20%-testing
data_train,data_val =train_test_split(data,test_size =0.2 ,random_state=42)

print(data_train.shape)
print(data_val.shape)


(800, 2)
(200, 2)


## Data Preprocessing

In [160]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [161]:
import re
from bs4 import BeautifulSoup, Comment


In [162]:
# Create a copy column 
data['preprocessed_text'] = data['text']


In [163]:
import warnings
from bs4 import MarkupResemblesLocatorWarning
# This line tells Python to completely ignore this specific Beautiful Soup warning
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

# --- 1. Remove inline JavaScript/CSS and HTML Comments ---
for i, text in enumerate(data['preprocessed_text']):
    soup = BeautifulSoup(text, "html.parser")
    
    # Remove all <script> and <style> tags
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

In [164]:
# Remove HTML comments
comments = soup.find_all(string=lambda t: isinstance(t, Comment))
for comment in comments:
        comment.extract()

# --- 2. Remove the remaining HTML tags and keep text only ---
data.iloc[i, data.columns.get_loc('preprocessed_text')] = soup.get_text()       

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [165]:
# Remove all the special characters ---                                                                    #pattern #empty string 
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x: re.sub(r'[^\w\s]', '', str(x)))

In [166]:
# Remove Numbers
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x:re.sub(r'[^\d+]','',str(x))) 

In [167]:
# Remove all single characters ---
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x: re.sub(r'\s+[a-zA-Z]\s+', ' ', str(x)))


In [168]:
#  Remove single characters from the start ---
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x: re.sub(r'^[a-zA-Z]\s+', ' ', str(x)))


In [169]:
#Remove prefixed 'b' ---
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x: re.sub(r'^b\s+', '', str(x)))

In [170]:
# Substitute multiple spaces with a single space ---
data['preprocessed_text'] = data['preprocessed_text'].apply(lambda x: re.sub(r'\s+', ' ', str(x)))

In [171]:
#Convert to Lowercase ---
data['preprocessed_text'] = data['preprocessed_text'].str.lower().str.strip()

## Now let's work on removing stopwords
Remove the stopwords.

In [172]:
import nltk
from nltk.corpus import stopwords
stop_words = stopwords.words('english')

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [173]:
from nltk.stem import WordNetLemmatizer

#Initialize the Lemmatizer tool
lemmatizer = WordNetLemmatizer()

# fucntion to Lemmatization
def stop_and_lemmatize(text):
    # Split the row of text into individual words
    words = str(text).split()
    
    # Keep the word ONLY if it is not a stopword, and transform it to its base form
    cleaned_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    
    # join the remaining filtered words back together into a single text sentence
    return " ".join(cleaned_words)

In [174]:
# Apply this function across your entire DataFrame column
data['preprocessed_text'] = data['preprocessed_text'].apply(stop_and_lemmatize)


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [175]:
from collections import Counter
import pandas as pd

In [192]:
# --- 1. Filter and count words for HAM (Normal Messages) ---

# Grab all text where the label is 0 (ham) and join it into one giant string
ham_text = " ".join(data[data['label'] == 0]['preprocessed_text'].astype(str))

# Split that giant string into individual words
ham_words = ham_text.split()

# Count the frequencies of each word
ham_word_counts = Counter(ham_words)
print(ham_word_counts)

Counter({'6': 12, '2010': 4, '1': 4, '1230': 2, '2026472972': 2, '1030': 2, '10': 2, '11': 2, '5': 2, '4': 2, '30': 2, '3': 2, '2026479071': 2, '13': 2, '34': 2, '66': 2, '100': 2, '1545': 1, '282009514': 1, '16': 1, '72010': 1, '61720101212666666666': 1, '1772009646': 1, '172120097266': 1, '02200911382': 1, '109': 1, '2026475548': 1, '12010114724120102130200870010112302050700000304085213060116012002009754050751921': 1, '142024191420': 1, '032009334': 1, '171620091001': 1, '20': 1, '622009850175': 1, '5132290': 1, '20264786332026475548': 1, '62010114620100606': 1, '182012338': 1, '17120091155': 1, '1996': 1, '172009539': 1, '2120091131221': 1, '7': 1, '20150484105739629051320150045554201504841057396290513201520211201215': 1, '171220121126371212': 1, '20100808': 1, '22200926': 1, '123220120264755482027364535': 1, '121320251421210420101141146114': 1, '52010541': 1, '272009650': 1, '282010503': 1, '17312010854': 1, '6745': 1, '072009408': 1, '27200965813': 1, '0220094046': 1, '202009812':

In [193]:
# --- 2. Filter and count words for SPAM (Fraudulent Messages) ---

# Grab all text where the label is 1 (spam) and join it into one giant string
spam_text = " ".join(data[data['label'] == 1]['preprocessed_text'].astype(str))

# Split that giant string into individual words
spam_words = spam_text.split()

# Count the frequencies of each word
spam_word_counts = Counter(spam_words)
print(spam_word_counts)

Counter({'10': 3, '30500000120': 3, '00': 3, '44704570518420447045705184': 2, '25705100': 2, '215215215201070': 2, '22222222222222222222222222': 2, '10123458': 2, '4952001': 2, '75100301060521511614314904911226': 2, '509': 2, '1944142005199219982000200420200414200530042540100': 2, '23310': 2, '36759000001001194414200519921998200020042020041420053004': 2, '23': 2, '1500000000199730601060': 1, '222222222829222829222221823222222240225222214222351976402280022164967682922': 1, '18611342511342835850500000028352523492726567': 1, '125265310100127300000010337503300001032606746649111074038447746': 1, '32003072033311867477163300339920030720406010003387057969744296569033462': 1, '50000000': 1, '2500000000712360400023324297270200233242972702196019603214124401857806142115928311139685': 1, '15200715212003592301060226765493429203': 1, '202022422220222029270000000022220202202220222006220': 1, '52783237849910930567302109989648425288': 1, '22222222222422222222211996224200622222829222222322205221222222222

In [178]:
print(data_train.columns)

Index(['text', 'label'], dtype='str')


## Extra features

In [179]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,money_mark,suspicious_words,text_len
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",1,0,0,106
535,I have not been able to reach oscar this am. W...,0,0,0,101
695,; Huma Abedin B6I'm checking with Pat on the 5...,0,0,0,141
557,I can have it announced here on Monday - can't...,0,0,0,52
836,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,1,0,1,1750


## How would work the Bag of Words with Count Vectorizer concept?

In [194]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize the vectorizer

# lowercase=True and stop_words='english' to clean up text automatically!
vectorizer = CountVectorizer(lowercase=True, stop_words='english')

#Teaching the vectorizer  vocabulary and transform the text column into numbers
X_train_bow = vectorizer.fit_transform(data_train['text'])

# Transform the validation data
X_val_bow = vectorizer.transform(data_val['text'])

# print matrix 
bow_df = pd.DataFrame(X_train_bow.toarray(), columns=vectorizer.get_feature_names_out())
bow_df.head()



,00,000,000000,00000e25,00000e251,00000eur,000066,0000ff,000m,000million,...,â½s,â½t,â½ta,â½te,â½tica,â½to,â½trangers,â½x60ã,â½x70,â½ã
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [181]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

# Load the vectorizer 
stop_words_list = list(stopwords.words('english'))
tfidf_vectorizer = TfidfVectorizer(lowercase=True, stop_words=stop_words_list)


In [182]:
#Vectorize the dataset (Fit and transform train text, only transform validation text)
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['text'])
X_val_tfidf = tfidf_vectorizer.transform(data_val['text'])

In [183]:
# Print the shape of the vectorized dataset
print("Shape of Training Vectorized Dataset:", X_train_tfidf.shape)
print("Shape of Validation Vectorized Dataset:", X_val_tfidf.shape)

Shape of Training Vectorized Dataset: (800, 23238)
Shape of Validation Vectorized Dataset: (200, 23238)


## And the Train a Classifier?

In [184]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


In [185]:
# target labels 
y_train = data_train['label']
y_val = data_val['label']

In [186]:
#Initialize the classifier (Using Naive Bayes)
classifier = MultinomialNB()

In [187]:
# Train the model using your training vectors and training labels
classifier.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[433.,367.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.61,-0.78]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 23238)","[[1.11,0. ,0. ,...,0. ,0. ,0. ], [3.7 ,6.98,0.07,...,0.03,0.03,0.35]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 23238)","[[ -9.37,-10.12,-10.12,...,-10.12,-10.12,-10.12], [ -8.65, -8.12,-10.13,...,-10.17,-10.17, -9.9 ]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,23238


In [188]:
#Predict classifications on the validation dataset
y_pred = classifier.predict(X_val_tfidf)

In [189]:
# EVALUATE PERFORMANCE
print(f"Validation Accuracy: {accuracy_score(y_val, y_pred) * 100:.2f}%\n")
print("Detailed Classification Report:")
print(classification_report(y_val, y_pred))

Validation Accuracy: 95.00%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96       125
           1       0.88      1.00      0.94        75

    accuracy                           0.95       200
   macro avg       0.94      0.96      0.95       200
weighted avg       0.96      0.95      0.95       200



### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [190]:
# Your code